# Phase 7: QLoRA fine-tuning of the intent parser (GPU)

Trains LoRA adapters on Qwen2.5-1.5B and/or 3B (4-bit base, fp16 compute) from `finetune/data/sft_train.jsonl`,
then parses every evaluation task with the trained adapter (zero-shot, same system prompt as training).
Scoring runs on the laptop with `eval/score_parsers.py`.

**Before running:** the SFT files must be in the repo (`finetune/data/sft_train.jsonl`, `sft_val.jsonl`, built with
`finetune/data_gen/build_sft.py`), or attached as a Kaggle dataset (set `SFT_DIR`).

**Settings:** GPU T4 x1, Internet on, secret `GITHUB_TOKEN` for a private repo. *Save Version → Save & Run All*.
Set `SMOKE = True` first: 20 steps per model (320 examples), to check memory, that the loss falls, and the time per
step (the cell prints an estimate for the full run). Output is streamed live: a new loss line every 2 steps.

Outputs to download: `ft_<model>.zip` (adapter, training log, run info) and `parse_ft_<model>.zip` (parses + `system.txt`).

In [1]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
SFT_DIR = "finetune/data/v2"   # data v2: v1 + review fixes (docs/open_issues.md 2026-10-31); None = finetune/data/
SMOKE = False              # True: 50 steps per model (check); False: full training
# (config file, system name used on the laptop, base model for parsing)
RUNS = [
    ("finetune/configs/qwen25_1_5b.yaml", "intent_ft_1_5b", "Qwen/Qwen2.5-1.5B-Instruct"),
    ("finetune/configs/qwen25_3b.yaml",   "intent_ft_3b",   "Qwen/Qwen2.5-3B-Instruct"),
]
PARSE_AFTER_TRAINING = True    # parse all evaluation tasks with each adapter (skipped in SMOKE mode)

In [2]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, json, time
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
if SFT_DIR:
    os.makedirs("finetune/data", exist_ok=True)
    for f in ("sft_train.jsonl", "sft_val.jsonl", "sft_stats.json"):
        if os.path.exists(f"{SFT_DIR}/{f}"):
            shutil.copy(f"{SFT_DIR}/{f}", f"finetune/data/{f}")
NEEDED = ["finetune/train_qlora.py", "eval/parse_contracts.py", "src/igt/agents/intent_parser.py",
          "finetune/data/sft_train.jsonl", "finetune/data/sft_val.jsonl"] + [r[0] for r in RUNS]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them, or set SFT_DIR): {missing}"
n_train = sum(1 for _ in open("finetune/data/sft_train.jsonl"))
n_val = sum(1 for _ in open("finetune/data/sft_val.jsonl"))
print(f"SFT data: {n_train} train, {n_val} val")

no GITHUB_TOKEN secret; cloning without a token (public repo)


Cloning into '/kaggle/working/repo'...


commit 8e42ab7607a6aa4abd0f1e8b09ccd3d47c2ecc48
SFT data: 3614 train, 796 val


In [3]:
# ---------------- Libraries ----------------
subprocess.run(["pip", "install", "-q", "-r", "finetune/requirements-train.txt"], check=True)
import torch, transformers, peft
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available()

GPU: Tesla T4
torch 2.11.0+cu128 | transformers 5.16.1 | peft 0.21.2


In [4]:
# ---------------- Train (and parse) each model; one subprocess per step frees GPU memory ----------------
def sh(cmd):
    """Run a step and stream its output live (the training loss is printed every logging step)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    assert p.wait() == 0, cmd
    return (time.time() - t0) / 60

for config, system, base in RUNS:
    tag = os.path.basename(config).replace(".yaml", "")
    out = f"finetune/runs/{tag}" + ("_smoke" if SMOKE else "")
    # Validation loss is only for monitoring: a fixed random 200 of the 796 val examples (64 in smoke mode),
    # so each evaluation takes minutes, not most of an hour. Parsing the evaluation tasks is separate.
    sets = [f"output_dir={out}"] + (["max_steps=20", "eval_steps=10", "logging_steps=2", "limit_val=64"] if SMOKE
                                    else ["limit_val=200"])
    print(f"===== train {system} ({config}) -> {out}")
    mins = sh(["python", "finetune/train_qlora.py", "--config", config, "--set", *sets])
    info = json.load(open(f"{out}/run_info.json"))
    print(f"trained in {mins:.1f} min | loss {info['first_logged_loss']} -> {info['last_logged_loss']} | eval {info['final_eval'].get('eval_loss')}")
    if SMOKE:   # full run = about 452 optimizer steps (3,614 examples x 2 epochs / 16)
        print(f"estimated full training for this model: ~{mins * 452 / info['steps'] / 60:.1f} h (+ evaluations and parsing)")
    open(f"{out}/git_commit.txt", "w").write(COMMIT)
    print("saved:", shutil.make_archive(f"/kaggle/working/ft_{tag}" + ("_smoke" if SMOKE else ""), "zip", out))
    if PARSE_AFTER_TRAINING and not SMOKE:
        pout = f"/kaggle/working/parse_ft_{tag}"
        print(f"===== parse with {system}")
        mins = sh(["python", "eval/parse_contracts.py", "--model", base, "--load-in-4bit",
                   "--adapter", f"{out}/adapter", "--out", pout, "--batch", "4"])
        open(f"{pout}/system.txt", "w").write(system)
        open(f"{pout}/git_commit.txt", "w").write(COMMIT)
        print(f"parsed in {mins:.1f} min; saved:", shutil.make_archive(pout, "zip", pout))

===== train intent_ft_1_5b (finetune/configs/qwen25_1_5b.yaml) -> finetune/runs/qwen25_1_5b
train_file: finetune/data/sft_train.jsonl sha256 4222330ebad0
val_file: finetune/data/sft_val.jsonl sha256 79e9db54a13f
train 3614 (dropped 0 too long) | val 200 (dropped 0) | tokens median 1611 max 1827

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 154.73it/s]
trainable params 18,464,768 of 907,081,216 (2.04%)

  0%|          | 0/226 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)

  4%|▍         | 10/226 [10:25<3:46:03, 62.79s/it]
 

In [5]:
# ---------------- Clean up (keep only the zips and outputs) ----------------
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
print(sorted(f for f in os.listdir("/kaggle/working") if f.endswith(".zip")))

['ft_qwen25_1_5b.zip', 'ft_qwen25_3b.zip', 'parse_ft_qwen25_1_5b.zip', 'parse_ft_qwen25_3b.zip']
